In [63]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
from esda import Moran
import geopandas as gpd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from libpysal.weights import KNN
from spreg import OLS, GM_Error_Het
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [64]:
property_old = gpd.read_file('output/property.gpkg')
canopy_norm = pd.read_csv('output/property_canopy_normalized.csv')

In [65]:
property_new = property_old.loc[
  :,
  ~property_old.columns.str.match(r"^canopy_\d+_\d+$")
]

In [66]:
property_full = property_new.merge(
    canopy_norm[[
      'property_id',
      'canopy_0_25',
      'canopy_25_50',
      'canopy_50_75',
      'canopy_75_100',
      'canopy_100_150',
      'canopy_150_200',
      'canopy_200_250',
      'canopy_250_300',
      'canopy_300_350',
      'canopy_350_400'
    ]],
    on='property_id',
    how='left'
)

In [67]:
property = property_full.drop(columns=['property_id', 'residuals'])

In [68]:
property['log_price'] = np.log(property['GrossSalePrice'])

In [69]:
property.to_file('output/property_all.gpkg')

In [55]:
property.columns

Index(['GrossSalePrice', 'AgeAtSale', 'LandArea', 'TotalFloorArea',
       'water_DIST', 'bus_DIST', 'Census_Pop', 'RnkIMDNoEm', 'RnkIMDNoIn',
       'RnkIMDNoCr', 'RnkIMDNoHo', 'RnkIMDNoHe', 'RnkIMDNoEd', 'RnkIMDNoAc',
       'DECILE_high', 'DECILE_prim', 'Median_Income', 'CBD_DIST',
       'cycleways_DIST', 'cycle_DENS', 'year_2018', 'year_2019', 'geometry',
       'canopy_0_25', 'canopy_25_50', 'canopy_50_75', 'canopy_75_100',
       'canopy_100_150', 'canopy_150_200', 'canopy_200_250', 'canopy_250_300',
       'canopy_300_350', 'canopy_350_400', 'log_price'],
      dtype='object')

In [29]:
w = KNN.from_dataframe(property, k=8)
w.transform = 'R'

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 12 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


In [30]:
y = property['log_price'].values.reshape(-1, 1)

In [34]:
non_x_cols = [
  # unneccessary columns
  'GrossSalePrice', 'geometry', 'log_price',
  # removed columns 
  # "AgeAtSale",  # include
  # "Census_Pop",
  # "RnkIMDNoEm",
  # "RnkIMDNoIn",
  # "RnkIMDNoCr",
  # "RnkIMDNoHo",
  # "RnkIMDNoHe",
  # "RnkIMDNoEd",
  # "RnkIMDNoAc",
  # "DECILE_high",
  # "Median_Income", # include
  "DECILE_prime",
  'TotalFloorArea',
  'LandArea',
  'water_DIST',
  'bus_DIST',
  'DECILE_prim',
  'CBD_DIST',
  'cycleways_DIST',
  'cycle_DENS',
  'year_2018',
  'year_2019',
  # 'canopy_0_25',
  # 'canopy_25_50',
  # 'canopy_50_75',
  # 'canopy_75_100',
  # 'canopy_100_150',
  # 'canopy_150_200',
  # 'canopy_200_250',
  # 'canopy_250_300',
  # 'canopy_300_350',
  # 'canopy_350_400'
]

x_cols = [col for col in property.columns if col not in non_x_cols]

for col in x_cols:
  print(col)

X = property.loc[:, x_cols]

X = X.values

AgeAtSale
Census_Pop
RnkIMDNoEm
RnkIMDNoIn
RnkIMDNoCr
RnkIMDNoHo
RnkIMDNoHe
RnkIMDNoEd
RnkIMDNoAc
DECILE_high
Median_Income
canopy_0_25
canopy_25_50
canopy_50_75
canopy_75_100
canopy_100_150
canopy_150_200
canopy_200_250
canopy_250_300
canopy_300_350
canopy_350_400


In [35]:
X = property.loc[:, x_cols]
X = X.values
sem = GM_Error_Het(y, X, w, name_y = 'log_price', name_x = x_cols)
print(sem.summary)

REGRESSION RESULTS
------------------

SUMMARY OF OUTPUT: GM SPATIALLY WEIGHTED LEAST SQUARES (HET)
------------------------------------------------------------
Data set            :     unknown
Weights matrix      :     unknown
Dependent Variable  :   log_price                Number of Observations:       12317
Mean dependent var  :     13.1505                Number of Variables   :          22
S.D. dependent var  :      0.3355                Degrees of Freedom    :       12295
Pseudo R-squared    :      0.4679
N. of iterations    :           1                Step1c computed       :          No

------------------------------------------------------------------------------------
            Variable     Coefficient       Std.Error     z-Statistic     Probability
------------------------------------------------------------------------------------
            CONSTANT        13.03902         0.04833       269.77547         0.00000
           AgeAtSale        -0.00280         0.00010    

In [ ]:
sem.lam

AttributeError: 'GM_Error_Het' object has no attribute 'lam'